# Classic ML Animal Classification: cattle vs sheep

Этот notebook нужен как мост между простым табличным Machine Learning и нейросетями из ВКР.

В ВКР классификация выглядит так:

```text
crop image животного -> CNN / VGG16 -> cattle или sheep
```

Здесь мы сделаем более простой вариант без изображений и нейросетей:

```text
табличные признаки животного -> classic ML model -> cattle или sheep
```

Главная учебная идея:

> В classic ML признаки задает человек. В CNN модель сама извлекает признаки из пикселей. Но логика classification остается одинаковой: `X -> model -> y_pred -> metrics`.

## Что будет в notebook

1. Создадим synthetic dataset `cattle/sheep`.
2. Разберем, что такое `X`, `y`, признаки и target.
3. Проведем EDA.
4. Сделаем train/validation/test split.
5. Построим preprocessing через `ColumnTransformer` и `Pipeline`.
6. Обучим Logistic Regression, Decision Tree, Random Forest, SVM.
7. Сравним accuracy, precision, recall, F1.
8. Построим confusion matrix.
9. Посмотрим feature importance.
10. Свяжем classic ML с CNN и YOLOv8.

## 1. Импорт библиотек

Нам нужны стандартные библиотеки для классического ML:

- `pandas` - таблицы;
- `numpy` - числа и генерация synthetic data;
- `matplotlib` и `seaborn` - графики;
- `scikit-learn` - split, preprocessing, модели и метрики.

В этом notebook нет PyTorch, CNN, YOLO и картинок. Это намеренно: сначала учим базовую идею classification.

In [ ]:
# numpy нужен для генерации чисел и случайного synthetic dataset.
import numpy as np

# pandas нужен для работы с табличными данными: строки, колонки, DataFrame.
import pandas as pd

# matplotlib нужен для базовых графиков.
import matplotlib.pyplot as plt

# seaborn делает статистические графики более аккуратными.
import seaborn as sns

# train_test_split делит dataset на train, validation и test.
from sklearn.model_selection import train_test_split

# ColumnTransformer позволяет отдельно обрабатывать числовые и категориальные признаки.
from sklearn.compose import ColumnTransformer

# Pipeline объединяет preprocessing и модель в один объект.
from sklearn.pipeline import Pipeline

# StandardScaler приводит числовые признаки к похожему масштабу.
from sklearn.preprocessing import StandardScaler

# OneHotEncoder переводит текстовые категории в числовые dummy-признаки.
from sklearn.preprocessing import OneHotEncoder

# LogisticRegression - базовая линейная модель для classification.
from sklearn.linear_model import LogisticRegression

# DecisionTreeClassifier - дерево решений, легко объяснять новичкам.
from sklearn.tree import DecisionTreeClassifier

# RandomForestClassifier - ансамбль деревьев, сильная classic ML модель.
from sklearn.ensemble import RandomForestClassifier

# SVC - Support Vector Classifier, classic ML модель с margin-идеей.
from sklearn.svm import SVC

# Метрики classification.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# Настраиваем стиль графиков.
sns.set_theme(style="whitegrid")

# Фиксируем random seed, чтобы dataset и результаты повторялись.
RANDOM_STATE = 42

# Создаем генератор случайных чисел numpy.
rng = np.random.default_rng(RANDOM_STATE)

## 2. Создаем synthetic dataset

У нас нет реальной таблицы с измерениями коров и овец, поэтому создадим учебный synthetic dataset.

Это нормально для обучения: мы сами задаем признаки так, чтобы они были понятными.

Признаки:

- `weight_kg` - вес животного;
- `height_cm` - высота;
- `body_length_cm` - длина тела;
- `wool_density` - плотность шерсти от 0 до 1;
- `horns` - есть ли рога;
- `color_group` - цветовая группа;
- `ear_shape` - форма ушей.

Target:

- `animal_class`: `cattle` или `sheep`.

Важно: это не биологически точный dataset. Он учебный, чтобы понять classification pipeline.

In [ ]:
# Количество synthetic examples для каждого класса.
n_cattle = 300
n_sheep = 300

# -----------------------------
# Генерируем числовые признаки cattle.
# -----------------------------

# Вес коров обычно сильно больше веса овец, поэтому берем среднее около 620 кг.
cattle_weight = rng.normal(loc=620, scale=80, size=n_cattle)

# Высота коров также больше: среднее около 145 см.
cattle_height = rng.normal(loc=145, scale=10, size=n_cattle)

# Длина тела коров: среднее около 220 см.
cattle_length = rng.normal(loc=220, scale=25, size=n_cattle)

# Плотность шерсти условно ниже: вокруг 0.25.
cattle_wool = rng.normal(loc=0.25, scale=0.10, size=n_cattle)

# Ограничиваем wool_density диапазоном от 0 до 1.
cattle_wool = np.clip(cattle_wool, 0, 1)

# -----------------------------
# Генерируем числовые признаки sheep.
# -----------------------------

# Вес овец меньше: среднее около 75 кг.
sheep_weight = rng.normal(loc=75, scale=15, size=n_sheep)

# Высота овец меньше: среднее около 75 см.
sheep_height = rng.normal(loc=75, scale=8, size=n_sheep)

# Длина тела овец: среднее около 110 см.
sheep_length = rng.normal(loc=110, scale=15, size=n_sheep)

# Плотность шерсти выше: вокруг 0.80.
sheep_wool = rng.normal(loc=0.80, scale=0.12, size=n_sheep)

# Ограничиваем wool_density диапазоном от 0 до 1.
sheep_wool = np.clip(sheep_wool, 0, 1)

# -----------------------------
# Генерируем категориальные признаки.
# -----------------------------

# Для cattle чаще встречаются brown/black/white группы.
cattle_color = rng.choice(["brown", "black", "white", "mixed"], size=n_cattle, p=[0.40, 0.30, 0.15, 0.15])

# Для sheep чаще встречаются white/mixed группы.
sheep_color = rng.choice(["white", "mixed", "black", "brown"], size=n_sheep, p=[0.60, 0.25, 0.10, 0.05])

# У cattle чаще условно rounded/long ears.
cattle_ears = rng.choice(["rounded", "long", "short"], size=n_cattle, p=[0.45, 0.40, 0.15])

# У sheep чаще short/rounded ears.
sheep_ears = rng.choice(["short", "rounded", "long"], size=n_sheep, p=[0.55, 0.35, 0.10])

# horns задаем как yes/no. У cattle в synthetic data рога чаще.
cattle_horns = rng.choice(["yes", "no"], size=n_cattle, p=[0.65, 0.35])

# У sheep в synthetic data рога реже.
sheep_horns = rng.choice(["yes", "no"], size=n_sheep, p=[0.20, 0.80])

# -----------------------------
# Собираем две таблицы: cattle и sheep.
# -----------------------------

# DataFrame для cattle.
cattle_df = pd.DataFrame({
    "weight_kg": cattle_weight,
    "height_cm": cattle_height,
    "body_length_cm": cattle_length,
    "wool_density": cattle_wool,
    "horns": cattle_horns,
    "color_group": cattle_color,
    "ear_shape": cattle_ears,
    "animal_class": "cattle",
})

# DataFrame для sheep.
sheep_df = pd.DataFrame({
    "weight_kg": sheep_weight,
    "height_cm": sheep_height,
    "body_length_cm": sheep_length,
    "wool_density": sheep_wool,
    "horns": sheep_horns,
    "color_group": sheep_color,
    "ear_shape": sheep_ears,
    "animal_class": "sheep",
})

# Объединяем cattle и sheep в один dataset.
dataset = pd.concat([cattle_df, sheep_df], ignore_index=True)

# Перемешиваем строки, чтобы сначала не шли все cattle, а потом все sheep.
dataset = dataset.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Округляем числовые колонки для более аккуратного отображения.
for col in ["weight_kg", "height_cm", "body_length_cm", "wool_density"]:
    dataset[col] = dataset[col].round(2)

# Показываем первые 10 строк dataset.
dataset.head(10)

## 3. Что такое X и y

В supervised classification у нас есть:

```text
X = признаки объекта
 y = правильный класс объекта
```

В этом notebook:

```text
X = weight_kg, height_cm, body_length_cm, wool_density, horns, color_group, ear_shape
 y = animal_class
```

Модель учится находить связь между признаками `X` и классом `y`.

In [ ]:
# X содержит все признаки, по которым модель будет делать prediction.
X = dataset.drop("animal_class", axis=1)

# y содержит правильный ответ: cattle или sheep.
y = dataset["animal_class"]

# Печатаем размер X: строки = животные, колонки = признаки.
print("X shape:", X.shape)

# Печатаем размер y: одна метка класса на каждую строку X.
print("y shape:", y.shape)

# Показываем первые строки признаков.
display(X.head())

# Показываем первые target labels.
display(y.head())

## 4. EDA: проверяем баланс классов

Перед обучением полезно проверить, сколько примеров каждого класса.

Если один класс сильно больше другого, accuracy может обманывать.

Например, если 95% объектов - `cattle`, модель может всегда говорить `cattle` и получить 95% accuracy, но она вообще не будет находить `sheep`.

In [ ]:
# value_counts считает количество строк каждого класса.
class_counts = dataset["animal_class"].value_counts()

# Печатаем class balance как таблицу.
print(class_counts)

# Создаем график размера 6 на 4 дюйма.
plt.figure(figsize=(6, 4))

# Строим barplot: по x класс, по y количество примеров.
sns.barplot(x=class_counts.index, y=class_counts.values)

# Добавляем заголовок.
plt.title("Class distribution")

# Подписываем оси.
plt.xlabel("Animal class")
plt.ylabel("Number of examples")

# Показываем график.
plt.show()

## 5. EDA: как признаки отличаются между cattle и sheep

Теперь посмотрим, как числовые признаки распределяются по классам.

Это важно для понимания classic ML:

- если признаки хорошо разделяют классы, модели будет легче;
- если признаки сильно пересекаются, задача сложнее;
- CNN делает похожую работу, но признаки извлекает сама из пикселей.

In [ ]:
# Список числовых признаков.
numeric_features = ["weight_kg", "height_cm", "body_length_cm", "wool_density"]

# Создаем сетку графиков 2 на 2.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Превращаем axes в плоский список, чтобы удобно итерироваться.
axes = axes.ravel()

# Для каждого числового признака строим boxplot по классам.
for ax, feature in zip(axes, numeric_features):
    # Boxplot показывает медиану, разброс и выбросы признака по каждому классу.
    sns.boxplot(data=dataset, x="animal_class", y=feature, ax=ax)
    
    # Добавляем понятный заголовок.
    ax.set_title(f"{feature} by class")

# Делаем layout аккуратнее.
plt.tight_layout()

# Показываем графики.
plt.show()

## 6. EDA: scatter plot двух важных признаков

Scatter plot помогает увидеть, разделяются ли классы в пространстве признаков.

Например, если `weight_kg` и `wool_density` хорошо разделяют cattle/sheep, даже простая модель сможет классифицировать животных.

In [ ]:
# Создаем график размера 8 на 5.
plt.figure(figsize=(8, 5))

# Каждая точка - одно животное.
# x = вес, y = плотность шерсти, color = класс.
sns.scatterplot(
    data=dataset,
    x="weight_kg",
    y="wool_density",
    hue="animal_class",
    alpha=0.75
)

# Добавляем заголовок.
plt.title("Weight vs wool density")

# Показываем график.
plt.show()

## 7. Train / validation / test split

Разделяем данные на три части:

```text
train       -> модель учится
validation  -> выбираем лучшую модель
 test       -> финальная честная проверка
```

Мы используем пропорции:

```text
train = 60%
validation = 20%
test = 20%
```

Параметр `stratify=y` сохраняет баланс `cattle/sheep` в каждом split.

In [ ]:
# Сначала отделяем test set: 20% всех данных.
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Потом оставшиеся 80% делим на train и validation.
# test_size=0.25 от 80% дает 20% от исходного dataset.
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

# Печатаем размеры split.
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)
print("Test:", X_test.shape, y_test.shape)

# Проверяем баланс классов в каждом split.
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True).round(3))

print("\nValidation class balance:")
print(y_valid.value_counts(normalize=True).round(3))

print("\nTest class balance:")
print(y_test.value_counts(normalize=True).round(3))

## 8. Preprocessing: числовые и категориальные признаки

Classic ML модели принимают числа.

У нас есть два типа признаков:

1. Числовые:
   - `weight_kg`
   - `height_cm`
   - `body_length_cm`
   - `wool_density`

2. Категориальные:
   - `horns`
   - `color_group`
   - `ear_shape`

Что делаем:

- числовые признаки масштабируем через `StandardScaler`;
- категориальные кодируем через `OneHotEncoder`.

Важно: preprocessing должен обучаться только на train, чтобы не было data leakage.

In [ ]:
# Находим числовые колонки по типам данных.
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Находим категориальные колонки по типу object.
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

# Печатаем найденные признаки.
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# Transformer для числовых признаков.
numeric_transformer = Pipeline(steps=[
    # StandardScaler вычитает среднее и делит на стандартное отклонение.
    ("scaler", StandardScaler()),
])

# Transformer для категориальных признаков.
categorical_transformer = Pipeline(steps=[
    # OneHotEncoder превращает категории в 0/1 колонки.
    # handle_unknown='ignore' защищает от ошибки, если на valid/test появится новая категория.
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# ColumnTransformer применяет разные transformers к разным колонкам.
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

## 9. Обучаем classic ML модели

Мы сравним четыре модели:

1. **Logistic Regression** - базовая линейная модель для classification.
2. **Decision Tree** - дерево решений; легко объяснять правилами `if/else`.
3. **Random Forest** - много деревьев, результат усредняется/голосуется.
4. **SVM / SVC** - ищет границу между классами с максимальным margin.

Каждую модель завернем в `Pipeline`:

```text
raw table -> preprocessing -> model -> prediction
```

In [ ]:
# Словарь candidate models: имя модели -> объект модели sklearn.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "SVM": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
}

# Здесь сохраним обученные pipelines.
fitted_pipelines = {}

# Здесь сохраним validation метрики.
validation_results = []

# Обучаем каждую модель по очереди.
for model_name, model in models.items():
    # Pipeline состоит из preprocessing и самой модели.
    pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model),
    ])
    
    # fit обучает preprocessing только на train и затем обучает модель.
    pipeline.fit(X_train, y_train)
    
    # predict делает предсказания на validation set.
    valid_pred = pipeline.predict(X_valid)
    
    # accuracy = доля правильных ответов.
    accuracy = accuracy_score(y_valid, valid_pred)
    
    # precision_macro = насколько точны предсказания каждого класса в среднем.
    precision = precision_score(y_valid, valid_pred, average="macro")
    
    # recall_macro = насколько хорошо модель находит каждый класс в среднем.
    recall = recall_score(y_valid, valid_pred, average="macro")
    
    # f1_macro = баланс precision и recall.
    f1 = f1_score(y_valid, valid_pred, average="macro")
    
    # Сохраняем результаты в список словарей.
    validation_results.append({
        "model": model_name,
        "valid_accuracy": accuracy,
        "valid_precision_macro": precision,
        "valid_recall_macro": recall,
        "valid_f1_macro": f1,
    })
    
    # Сохраняем обученный pipeline, чтобы потом взять лучшую модель.
    fitted_pipelines[model_name] = pipeline

# Превращаем результаты в DataFrame и сортируем по F1.
validation_results_df = pd.DataFrame(validation_results).sort_values("valid_f1_macro", ascending=False)

# Показываем validation leaderboard.
validation_results_df

## 10. Выбираем лучшую модель и оцениваем test set

Validation нужен для выбора модели.

Test нужен для финальной честной проверки.

Мы выбираем модель с максимальным `valid_f1_macro`, а потом один раз оцениваем ее на `test`.

In [ ]:
# Берем имя лучшей модели из первой строки leaderboard.
best_model_name = validation_results_df.iloc[0]["model"]

# Достаем обученный pipeline лучшей модели.
best_pipeline = fitted_pipelines[best_model_name]

# Делаем prediction на test set.
test_pred = best_pipeline.predict(X_test)

# Считаем test metrics.
test_metrics = {
    "model": best_model_name,
    "test_accuracy": accuracy_score(y_test, test_pred),
    "test_precision_macro": precision_score(y_test, test_pred, average="macro"),
    "test_recall_macro": recall_score(y_test, test_pred, average="macro"),
    "test_f1_macro": f1_score(y_test, test_pred, average="macro"),
}

# Печатаем лучшую модель.
print("Best model selected on validation:", best_model_name)

# Показываем test metrics как таблицу.
pd.DataFrame([test_metrics])

## 11. Classification report

`classification_report` показывает метрики отдельно по каждому классу.

Это полезно, потому что модель может хорошо находить `cattle`, но хуже находить `sheep`, или наоборот.

Для защиты важно уметь объяснить:

- `precision`;
- `recall`;
- `F1-score`;
- `support`.

In [ ]:
# classification_report печатает precision, recall, f1-score и support по каждому классу.
print(classification_report(y_test, test_pred))

## 12. Confusion matrix

Confusion matrix показывает, где модель ошибается.

Для binary classification `cattle/sheep` она отвечает на вопросы:

- сколько cattle предсказаны как cattle;
- сколько cattle перепутаны со sheep;
- сколько sheep предсказаны как sheep;
- сколько sheep перепутаны с cattle.

В ВКР такие матрицы нужны для CNN, VGG16, ResNet50, MobileNetV2.

In [ ]:
# Задаем порядок классов, чтобы строки/колонки были стабильными.
labels = ["cattle", "sheep"]

# Считаем confusion matrix.
cm = confusion_matrix(y_test, test_pred, labels=labels)

# Создаем график.
plt.figure(figsize=(6, 5))

# Рисуем heatmap confusion matrix.
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

# Подписываем оси.
plt.xlabel("Predicted class")
plt.ylabel("True class")

# Добавляем заголовок.
plt.title(f"Confusion matrix: {best_model_name}")

# Показываем график.
plt.show()

## 13. Feature importance: какие признаки важны

Classic ML часто легче объяснять, потому что признаки уже понятные человеку.

Для `RandomForestClassifier` можно посмотреть `feature_importances_`.

Это отвечает на вопрос:

> Какие признаки сильнее всего помогали модели отличать cattle от sheep?

В CNN аналог сложнее: там модель сама строит признаки из пикселей, поэтому мы используем Grad-CAM, чтобы понять, куда она смотрит.

In [ ]:
# Создаем отдельный Random Forest pipeline для feature importance.
rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
])

# Обучаем Random Forest на train set.
rf_pipeline.fit(X_train, y_train)

# Достаем обученный preprocessing шаг.
trained_preprocessor = rf_pipeline.named_steps["preprocess"]

# Получаем имена признаков после preprocessing.
feature_names = trained_preprocessor.get_feature_names_out()

# Достаем обученную Random Forest модель.
rf_model = rf_pipeline.named_steps["model"]

# feature_importances_ показывает относительную важность каждого признака.
importances = rf_model.feature_importances_

# Собираем feature importance в таблицу.
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False)

# Показываем топ-10 признаков.
importance_df.head(10)

In [ ]:
# Берем топ-10 признаков по важности.
top_importance = importance_df.head(10)

# Создаем график.
plt.figure(figsize=(9, 5))

# Рисуем horizontal barplot.
sns.barplot(data=top_importance, x="importance", y="feature")

# Добавляем заголовок.
plt.title("Top feature importances from Random Forest")

# Показываем график.
plt.show()

## 14. Prediction для нового животного

Теперь создадим пример нового животного и попросим модель определить класс.

Это показывает полный путь inference:

```text
новые признаки -> preprocessing -> model -> predicted class
```

В CNN аналогично:

```text
новый crop image -> transforms -> CNN -> predicted class
```

In [ ]:
# Создаем DataFrame с одним новым животным.
new_animal = pd.DataFrame([{
    "weight_kg": 82,
    "height_cm": 78,
    "body_length_cm": 115,
    "wool_density": 0.88,
    "horns": "no",
    "color_group": "white",
    "ear_shape": "short",
}])

# Показываем признаки нового животного.
display(new_animal)

# Делаем prediction лучшей моделью.
predicted_class = best_pipeline.predict(new_animal)[0]

# Если модель умеет predict_proba, покажем вероятности классов.
if hasattr(best_pipeline.named_steps["model"], "predict_proba"):
    probabilities = best_pipeline.predict_proba(new_animal)[0]
    class_names = best_pipeline.named_steps["model"].classes_
    proba_table = pd.DataFrame({"class": class_names, "probability": probabilities})
    display(proba_table)

# Печатаем итоговый predicted class.
print("Predicted class:", predicted_class)

## 15. Мост к CNN и YOLOv8

Теперь можно объяснить переход к ВКР.

### Classic ML classification

```text
табличные признаки животного -> Logistic Regression / Random Forest / SVM -> cattle или sheep
```

Человек заранее задает признаки:

- вес;
- высота;
- длина тела;
- плотность шерсти;
- цвет;
- форма ушей.

### CNN image classification

```text
crop image животного -> CNN / VGG16 / ResNet50 / MobileNetV2 -> cattle или sheep
```

Человек уже не задает признаки вручную. CNN сама извлекает признаки из пикселей:

- края;
- текстуры;
- форму тела;
- шерсть;
- визуальные паттерны.

### YOLOv8 object detection

```text
полное изображение -> YOLOv8 -> bounding boxes + class labels -> counting
```

YOLO добавляет еще одну задачу: не только классифицировать животное, но и найти, где оно находится на полном изображении.

## Главное для защиты

Можно сказать так:

> Classic ML помогает понять базовую задачу classification: есть признаки X, есть класс y, модель учится отличать cattle от sheep и оценивается через accuracy, precision, recall и F1. В CNN признаки уже не задаются вручную: модель извлекает их из изображения. А YOLOv8 идет дальше: она ищет животных на полном кадре и позволяет считать их количество.

## 16. Что нужно запомнить

1. `X` - признаки объекта.
2. `y` - правильный класс.
3. `train` - данные для обучения модели.
4. `validation` - данные для выбора лучшей модели.
5. `test` - финальная честная проверка.
6. `accuracy` - доля правильных ответов.
7. `precision` - насколько чисты предсказания класса.
8. `recall` - сколько реальных объектов класса найдено.
9. `F1` - баланс precision и recall.
10. `confusion matrix` - таблица ошибок между классами.
11. Classic ML использует human-designed features.
12. CNN учит признаки из pixels.
13. YOLO делает detection на full image и позволяет counting.